In [ ]:
import sys

if sys.version_info[0] + 0.1 * sys.version_info[1] < 3.6:
    raise Exception("Must be using Python 3.6+ version")

In [ ]:
%matplotlib inline
sys.path.append("..")

from plotly.subplots import make_subplots
import plotly.graph_objects as go

import numpy as np

import wavelet_transform.utils.training_data as train
from wavelet_transform.postprocess import plotting as plot
import wavelet_transform.utils.PCD_data as pcd
import wavelet_transform.wpt.inverse_transform as inv
from wavelet_transform.utils.extensions import fft, psd, to_dB, eng_form_plot #, trim_data
from wavelet_transform.utils.machine_learning import fit_and_evaluate, train_k_means, assess_parameter
import wavelet_transform.utils.kmeans_results as results
from wavelet_transform.utils.preprocessing import preprocess
from wavelet_transform.utils.save_plots import save_figure

result_id is used to store the results in a file that can be found using results.get_data(result_id)

In [ ]:
saveflag = False

In [ ]:
# PCD DATA
# "5-10", "5-34", "6-12" are the valid arguments for the 1st dataset
# tuples containing the voltage and frequency is used for the 2nd dataset
# e.g. (80, 1) for 80 mVpp, c1hat
frequency = 1.1 * 1e6
pcd_number = 1
voltage = 80
data, t_plot = pcd.get_data((voltage, pcd_number))  # [80, 120, 200, 280], [1, 5, 10]

no_clusters = (3, 6)
result_id = f"{voltage}_{pcd_number}_{no_clusters}"

In [ ]:
# # data from 16/11/23
# frequency = 2e6
# exp_number = 36  # currently testing 9 (350 mVpp), 15 (10? mVpp), 55 (350 mVpp transient)
# data, t_plot= pcd.get_data(exp_number)
# result_id = str(exp_number)

- test 2MHz with 400, 450 amplitude and high wave periods per pulse for interesting signals
- write up numerical studies and ga data plots

In [ ]:
# parameters = {
#     "amplitude": 200,
#     "frequency": "2",
#     "duty cycle": 10,
#     "prf": 20,  # pulse repetition frequency
#     "experiment": "00002",
# }

# if parameters["frequency"]=="2":
#     frequency = 2e6 #parameters["frequency"] * 1e6
# else:
#     frequency = 7.37e6
    
# dfbs = pcd.get_data(parameters)
# data, t_plot = dfbs["c2"]
# excitation_data, t_plot_ext = dfbs["c1"]

# result_id = (parameters["amplitude"], parameters["frequency"])

In [7]:
sampling_rate = np.round(1 / (t_plot[1] - t_plot[0]))
# freq_threshold = 20e6  # the nodes with norm_freq_range above this threshold will not be kept
# freq_threshold /= (sampling_rate / 2)
freq_threshold = 1
print(sampling_rate)

In [ ]:
# indx = np.abs(excitation_data) > 0.001
# indx = np.where(indx != 0)[0][0]
# pulse_rep_length = int(sampling_rate * (1 / (parameters["prf"] * 1e3)))
# indx_end = indx + pulse_rep_length
# data_segments = []
# excitation_data_segments = []

# while indx_end < len(data):
#     data_segments.append(data[indx:indx_end])
#     excitation_data_segments.append(excitation_data[indx:indx_end])
#     indx = indx_end
#     indx_end = indx_end + pulse_rep_length

In [ ]:
# data_segments[0].shape

In [ ]:
# %matplotlib ipympl
# import matplotlib.pyplot as plt
# plt.plot([np.linspace(1,len(data_segments[0]),len(data_segments[0])) for i in range(len(data_segments))],np.vstack(data_segments),'-')
# plt.show()

In [ ]:
# Parameters for the wavelet transform
wavelet = "db20"
levels = 12  # set to None to use max-level transform
mode = "symmetric"
use_min_entropy = True
min_level = 5

In [ ]:
# Plot PCD data and the excitation sound wave
# fft_lim = 5
# fig = make_subplots(rows=1, cols=1)
# fig.add_trace( go.Scatter(x=t_plot, y=data, mode='lines',name='data'),
#               row=1,col=1)
# # fig.add_trace( go.Scatter(x=t_plot, y=excitation_data ,mode='lines', name = 'excitation data'),
# #               row=1,col=1)

# fig.update_layout(height=600, width=1000,
#                   xaxis1 = dict(title='Time [s]'),
#                   yaxis1 = dict(title='Amplitude [DL]'),)
# fig.show()

In [ ]:
# Normalise
data_processed = preprocess(data, window=True) # [preprocess(data_segments[i], window=True) for i in range(len(data_segments))]

In [ ]:
# Trim the array down to a start and end time
# t_lims = (0e-6, 5e-6)
# data_processed, t_plot = trim_data(data_processed, t_plot, t_lims, return_time=True)

In [ ]:
# Plot generated data
# fft_data, freq = fft(data_processed, sampling_rate, frequency)
fft_lim = 10
def plot_psd(data, t):
    psd_data, freq = psd(data, sampling_rate, frequency)

    # t_plot = [i*10**6 for i in t]
    
    # plot sound wave and fft
    fig = make_subplots(rows=1, cols=1)
    # fig.add_trace(
    #     go.Scatter(x=t_plot, y=data, mode="lines", name="data"), row=1, col=1
    # )
    fig.add_trace(
        go.Scatter(x=freq, y=to_dB(psd_data), mode="lines", name="fft - amp"),
        row=1,
        col=1,
    )
    # fig.add_trace( go.Scatter(x=freq, y=np.angle(fft_data, deg=True),mode='lines', name = 'fft - phase'),
    #               row=3,col=1)
    
    fig.update_layout(
        height=500,
        width=1000,
        font_size=20,
        # xaxis1=dict(title="Time [&mu;s]"),
        xaxis1=dict(title="Freq [f/f<sub>0</sub>]", range=[0, fft_lim]),
        # xaxis3=dict(title="Freq [f/f<sub>0</sub>]", range=[0, fft_lim]),
        # yaxis1=dict(title="Amplitude [DL]"),
        yaxis1=dict(title="Amplitude [dB]", range=[-40, 40]), 
        # yaxis3=dict(title="Amplitude [rad]"), 
        legend=dict(x=0.8, y=1, font=dict(size=20)),
        plot_bgcolor="white",     
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')        
    fig.show()
    return fig

In [ ]:
fig = plot_psd(data_processed, t_plot)
save_figure(fig, f"{result_id}_data")

## Wavelet transform

In [ ]:
wpt_df, tr_obj = train.generate_training_data(
    data_processed, levels, wavelet, mode, 
    include_tr_obj=True, min_entropy=use_min_entropy,
    min_level=min_level, freq_threshold=freq_threshold
)

## Training

In [ ]:
# k-means parameters
kmeans_params = {
    "init": "k-means++",
    "n_init":10,
    "max_iter":100,
    "tol":1e-8,
    "verbose":0,
    "algorithm":"lloyd",
}

# training data
classifying_metrics = [
    "log average energy",
    "kurtosis",
    # "crest factor",
    # "level entropy",
    "spectral entropy",
    "average energy",
    # "skewness"
]
# Set to no_clusters to None to select optimal number of clusters by silhouette score
use_inverse_metrics = False
scale_data = "standard"
if use_inverse_metrics is True:
    inv_tr_obj = inv.InverseTransform(tr_obj)  # Use info from inverse transform object instead
    wpt_df_train = inv_tr_obj.info(leaf_nodes=True, freq_threshold=freq_threshold)
else:
    wpt_df_train = wpt_df.copy()

In [ ]:
data_train, labels, data_train_normalised, centers, scaler, kmeans = train_k_means(
    wpt_df_train, 
    classifying_metrics, 
    no_clusters=no_clusters, 
    kmeans_params=kmeans_params, 
    scale_data=scale_data
)
centers_scaled = scaler.fit_transform(centers)

In [ ]:
# fit_and_evaluate(kmeans, data_train_normalised, labels, name='wpt data', n_runs=100)

In [ ]:
# parameter_of_interest = "max_iter"
# values = [i for i in range(1, 100, 2)]
# rand_indices = assess_parameter(
#     parameter_of_interest, values, wpt_df_train, 
#     classifying_metrics, no_clusters
# )

# plt.plot(values, rand_indices)
# plt.title(f"Effect of {parameter_of_interest} on clustering algorithm")
# plt.ylabel("Adjusted rand index")
# plt.xlabel(f"{parameter_of_interest.capitalize()}")
# plt.show()

## Save results

In [ ]:
if saveflag == True:
    additional_params = {
        "raw_data": data,
        "driving frequency": frequency,
    }

    results.save_data(
        result_id, data_train, tr_obj, centers, labels, 
        sampling_rate, freq_threshold, **additional_params
    )

## Plot results

In [ ]:
fig = plot.plot_clustering(
    data_train, 
    centers, 
    labels, 
    [classifying_metrics[i] for i in [0, 1, 2]], 
    title=None
)

In [ ]:
# Plot centroid metrics
# xaxis_ticks_labels = dict((i, f"Clu÷ster {i}") for i in range(len(centers)))
fig = plot.plot_centers(classifying_metrics, centers) #, label_map=xaxis_ticks_labels)

In [ ]:
tr_obj.calc_frequencies(sampling_rate)
fig = plot.heisenberg(
    tr_obj, t_plot[-1], max_freq=freq_threshold, padding="repeat", use_log_scale=True
)

tr_obj.plot_tree(
    node_size="average energy",
    node_colour=list(labels), 
    freq_threshold=freq_threshold, 
    order="natural",
    sampling_rate=sampling_rate,
    show_frequency=False,
)

In [ ]:
# Set clusters to None to plot all of them, otherwise, plot a list of int
fig = plot.plot_fft_by_group(
    tr_obj, labels, sampling_rate,
    clusters=None, xlim=10, fundamental_freq=frequency,
    freq_threshold=freq_threshold, fig_height=350,
    plot_full_reconstruction=False    
)

fig = plot.plot_reconstructions_by_group(
    tr_obj, labels, t_plot, freq_threshold
)

In [ ]:
from wavelet_transform.postprocess.plotting import get_reconstructions_by_group
import plotly.graph_objects as go
recs = get_reconstructions_by_group(tr_obj, labels, freq_threshold)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=sum(recs),
        mode="lines",
        name="Signal with freq thresholding",
    )
)
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=tr_obj.data,
        mode="lines",
        name="Orignial signal"
    )
)


In [ ]:
import wavelet_transform.wpt.inverse_transform as inv
import matplotlib.pyplot as plt

# Add number to plot_centres to plot reconstructions of all nodes belonging to the cluster
plot_centres = []
for i, (node, assignment) in enumerate(zip(tr_obj.get_leaf_nodes(freq_threshold=freq_threshold), labels)):
    if assignment in plot_centres:
        print("spectral entropy:", node.spectral_entropy)
        rec = inv.inv_dwpt(tr_obj, node.level, nodes=node)
        psd_rec, freq = psd(rec, sampling_rate, frequency)
        # freq /= frequency
        plt.plot(frequency, to_dB(psd_rec))
        plt.xlim(0, 10)
        plt.title(f"frequency range: {np.divide(node.frequency_range,frequency)} (normalised by f0)")
        plt.show()
        plt.plot(rec)
        plt.show()

In [ ]:
# [phi, psi, x] = pywt.Wavelet('db2').wavefun(level=4)

In [ ]:
# import numpy as np
# import scipy.signal as sig
# win_func = sig.windows.blackmanharris(len(data_segments[2]), sym=False)

In [ ]:
# np.std(data_segments[2]*win_func)